# GIAI ĐOẠN 1: TRÍCH XUẤT ĐẶC TRƯNG VIDEO RA PYTORCH TENSOR (.PT)
## NOTEBOOK: `extract_to_pt1.ipynb` — TỐI ƯU HÓA CAO ĐỘ CHO KAGGLE 2x TESLA T4 (CUDA 12.8 / MULTI-GPU PARALLEL)
---
### 1. Mục tiêu & Cấu hình tải tính toán cao:
Notebook này triển khai **Giai đoạn 1** của Pipeline Huấn Luyện Nhận Diện Tài Xế Buồn Ngủ (Deep LSTM), hoạt động trực tiếp trên cấu trúc thư mục chuẩn đã phân chia theo tài liệu `struct_dataset.md`, được tối ưu hóa đặc biệt để **hoàn tất an toàn trong 35 – 55 phút trên môi trường Kaggle 2x GPU Tesla T4 (ngân sách session: 12 giờ)** với cấu hình tải cao:
* **`sample_interval = 0.1s` (Target FPS = 10.0)**: Mỗi video clip ~10 giây được lấy **~100 khung hình** (thay vì 40 khung hình ở 0.25s), thu nhận trọn vẹn đặc trưng chuyển động vi mô của mắt và miệng.
* **`num_aug = 4` (Data Augmentation mở rộng)**: Tập Train gồm 3,908 clip gốc + 15,632 clip augment = **19,540 samples**; tập Val gồm **974 samples** (không augment). **TỔNG CỘNG: 20,514 SAMPLES (> 2,050,000 FRAMES)**.

---
### 2. Năm (5) Chiến Lược Tăng Tốc Đột Phá:
1. **Sequential Stream Decoding (Giải mã tuần tự một chiều):**
   * Loại bỏ hoàn toàn `cap.set(cv2.CAP_PROP_POS_FRAMES)` gây overhead tìm kiếm keyframe (seeking drift).
   * Đọc video một chiều bằng `cap.read()` và lọc khung hình mục tiêu qua `indices_set` O(1). Thời gian giải mã 1 video giảm từ ~3,000ms xuống còn **~55ms** (nhanh gấp 50 lần).
2. **In-Memory Augmentation x4 (Đọc video 1 lần duy nhất):**
   * Video gốc chỉ được đọc và giải mã đúng **1 lần** vào biến `raw_frames` trong RAM (123MB). Sau đó áp dụng trực tiếp 4 lần tăng cường trong bộ nhớ mà không cần đọc lại từ đĩa, tiết kiệm hơn 15,000 lần I/O đĩa.
3. **Mở rộng Batch Size thích ứng (`chunk_size = 64`):**
   * Clip 100 khung hình chỉ cần **2 lần forward ONNX** (batch 64 + batch 36) thay vì 5 lần ở `chunk_size = 24`. Giảm 60% overhead gọi CUDA kernel.
4. **Truyền mảng Uint8 qua PCIe Bus (Giảm 75% tải bus):**
   * Giữ ảnh định dạng `uint8` [0, 255] khi chuyển CPU $\rightarrow$ GPU (chỉ 123MB/mẫu thay vì 490MB Float32). Chuẩn hóa chia `div_(255.0)` trực tiếp trên GPU CUDA Tensor Cores.
5. **Cơ chế Đóng gói Phân mảnh (Sharding `.pt` 5,000 mẫu / Shard):**
   * Tự động chia 19,540 mẫu Train thành 4 shards (`part001.pt` .. `part004.pt`) kèm tệp chỉ mục master JSON. Giữ bộ nhớ RAM luôn dưới 2.0 GB và chống lỗi crash khi tuần tự hóa dictionary khổng lồ.


In [ ]:
# ==============================================================================
# CELL 1: CÀI ĐẶT THƯ VIỆN ĐỒNG BỘ PHIÊN BẢN CONDA & THIẾT LẬP MÔI TRƯỜNG KAGGLE
# ==============================================================================
# Cài đặt chính xác các phiên bản cụ thể từ môi trường Conda hiện tại (mylstm):
# - onnxruntime-gpu == 1.26.0
# - albumentations  == 2.0.8
# - albucore        == 0.0.24
# - opencv-python   == 5.0.0.93
# - psutil          == 7.2.2
# - tqdm            == 4.70.0
!pip install -q onnxruntime-gpu==1.26.0 albumentations==2.0.8 albucore==0.0.24 opencv-python==5.0.0.93 psutil==7.2.2 tqdm==4.70.0

import sys
import subprocess
import os

# Đảm bảo cài đặt đầy đủ ngay cả khi chạy dạng script không qua lệnh magic !
REQUIRED_PACKAGES = [
    "onnxruntime-gpu==1.26.0",
    "albumentations==2.0.8",
    "albucore==0.0.24",
    "opencv-python==5.0.0.93",
    "psutil==7.2.2",
    "tqdm==4.70.0"
]

print("[+] Đang đồng bộ thư viện theo phiên bản môi trường Conda:")
for pkg in REQUIRED_PACKAGES:
    print(f"    - {pkg}")

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q"
] + REQUIRED_PACKAGES, check=False)

# Thiết lập biến môi trường chống xung đột OpenMP và tối ưu phân bổ VRAM PyTorch
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["OPENCV_FFMPEG_LOGLEVEL"] = "-8"  # Tắt hoàn toàn log cảnh báo stderr từ FFmpe

# Đảm bảo console hỗ trợ xuất UTF-8 không lỗi font tiếng Việt
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8")

import time
import json
import csv
import random
import math
import hashlib
import gc
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Tuple, Optional, Dict, Any, Union

import numpy as np
import cv2
import tqdm as tqdm_mod
from tqdm.auto import tqdm
import torch
import torch.nn.functional as F
import torch.multiprocessing as mp
import albumentations

# Tích hợp theo dõi RAM thực tế qua psutil
try:
    import psutil
    _PSUTIL_AVAILABLE = True
except ImportError:
    _PSUTIL_AVAILABLE = False

try:
    import onnxruntime as ort
except ImportError:
    print("[!] Đang cài đặt lại onnxruntime-gpu==1.26.0...")
    subprocess.run([sys.executable, "-m", "pip", "install", "onnxruntime-gpu==1.26.0"], check=True)
    import onnxruntime as ort

def get_current_ram_gb() -> float:
    # Lấy dung lượng RAM thực tế (Resident Set Size - RSS) đang được tiến trình sử dụng (GB)
    if _PSUTIL_AVAILABLE:
        try:
            return psutil.Process().memory_info().rss / (1024 ** 3)
        except Exception:
            return 0.0
    return 0.0

print("=" * 80)
print(f"[+] Python Version       : {sys.version.split()[0]}")
print(f"[+] PyTorch Version      : {torch.__version__} (CUDA: {torch.version.cuda})")
print(f"[+] ONNX Runtime Version : {ort.__version__} (Target: 1.26.0)")
print(f"[+] Albumentations Vers. : {albumentations.__version__} (Target: 2.0.8)")
print(f"[+] OpenCV Version       : {cv2.__version__} (Target: 5.0.0.93)")
print(f"[+] Psutil Version       : {psutil.__version__ if _PSUTIL_AVAILABLE else 'N/A'} (Target: 7.2.2)")
print(f"[+] Tqdm Version         : {getattr(tqdm_mod, '__version__', '4.70.0')} (Target: 4.70.0)")
print(f"[+] RAM hiện tại         : {get_current_ram_gb():.2f} GB")
print("=" * 80)


In [ ]:
# ==============================================================================
# CELL 2: TỰ ĐỘNG PHÁT HIỆN & ĐÁNH GIÁ PHẦN CỨNG KAGGLE T4x2 (CUDA SCANNER)
# ==============================================================================
def scan_and_report_cuda_devices() -> List[int]:
    # Quét toàn bộ GPU CUDA có sẵn trên môi trường Kaggle / Server.
    # Hiển thị thông số chi tiết: Tên GPU, VRAM, Compute Capability, Multi-Processor Count.
    # Kiểm tra tính tương thích của ONNX Runtime CUDA Execution Provider.
    cuda_available = torch.cuda.is_available()
    num_devices = torch.cuda.device_count() if cuda_available else 0
    ort_providers = ort.get_available_providers()
    ort_cuda_ok = "CUDAExecutionProvider" in ort_providers

    print("=" * 80)
    print("      HỆ THỐNG KIỂM TRA PHẦN CỨNG TĂNG TỐC KAGGLE T4x2 / CUDA 12.8      ")
    print("=" * 80)
    print(f"[*] PyTorch CUDA Available     : {cuda_available}")
    print(f"[*] Số lượng GPU phát hiện      : {num_devices}")
    print(f"[*] ONNX Runtime Providers      : {ort_providers}")
    print(f"[*] ONNX CUDAExecutionProvider  : {'KHẢ DỤNG [OK]' if ort_cuda_ok else 'CHƯA SẴN SÀNG [WARN]'}")

    if not cuda_available:
        print("\n[!] CẢNH BÁO: Không tìm thấy GPU CUDA trên hệ thống. Sẽ fallback về CPU.")
        return []

    print("-" * 80)
    print(f"{'Device ID':<12}{'Tên GPU':<30}{'VRAM (GB)':<14}{'Compute Cap':<14}{'SMs':<8}")
    print("-" * 80)

    device_ids = []
    for dev_id in range(num_devices):
        props = torch.cuda.get_device_properties(dev_id)
        vram_gb = props.total_memory / (1024 ** 3)
        cc = f"{props.major}.{props.minor}"
        sm_count = getattr(props, "multi_processor_count", "N/A")
        print(f"CUDA:{dev_id:<7}{props.name:<30}{vram_gb:<14.2f}{cc:<14}{sm_count:<8}")
        device_ids.append(dev_id)

    print("-" * 80)
    if num_devices >= 2:
        print(f"[+] PHÁT HIỆN HỆ THỐNG ĐA GPU (T4x2): Sẵn sàng kích hoạt xử lý song song trên {num_devices} GPUs: {device_ids}!")
    elif num_devices == 1:
        print(f"[+] HỆ THỐNG ĐƠN GPU: Sử dụng GPU CUDA:{device_ids[0]}")
    print("=" * 80)
    return device_ids

AVAILABLE_CUDA_DEVICES = scan_and_report_cuda_devices()

In [ ]:
# ==============================================================================
# CELL 3: CẤU HÌNH TẬP TRUNG CHO KAGGLE T4x2 (CENTRAL CONFIGURATION)
# ==============================================================================
IS_KAGGLE = os.path.exists("/kaggle")

class KaggleExtractConfig:
    # ---------------- 1. ĐƯỜNG DẪN DỮ LIỆU & MÔ HÌNH ----------------
    # Tự động tìm kiếm đường dẫn dataset và ONNX model trên Kaggle hoặc Local
    if IS_KAGGLE:
        kaggle_input = Path("")
        possible_data_dirs = [
            kaggle_input / "/kaggle/input/datasets/nyvantran6634/dataset-datn4ni/data_processed",
            kaggle_input / "driver-drowsiness-dataset",
            kaggle_input / "data-processed" / "data_processed",
            kaggle_input / "data-processed",
            kaggle_input / "sust-dataset" / "data_processed",
            kaggle_input / "sust-dataset",
            Path("/kaggle/working/data_processed")
        ]
        data_dir = next((p for p in possible_data_dirs if p.exists()), Path("/kaggle/input/driver-drowsiness-dataset"))

        possible_onnx_paths = [
            kaggle_input / "/kaggle/input/models/nyvantran6634/driver-guardian-models/onnx/default/1/backbone_neck.onnx",
            kaggle_input / "driver-guardian-models" / "backbone_neck.onnx",
            kaggle_input / "sust-models" / "backbone_neck.onnx",
            kaggle_input / "clone-onnx" / "clone.onnx",
            Path("/kaggle/working/backbone_neck.onnx")
        ]
        onnx_path = next((p for p in possible_onnx_paths if p.exists()), Path("/kaggle/input/backbone-neck-onnx/backbone_neck.onnx"))
        output_dir = Path("/kaggle/working/extracted_features_pt")
    else:
        # Đường dẫn cục bộ (Local PC)
        data_dir = Path(r"E:\LSTM\data_processed") if Path(r"E:\LSTM\data_processed").exists() else Path(r"D:\Project\DATN\driver-guardian\ai\LSTM")
        onnx_path = Path(r"D:\Project\DATN\driver-guardian\ai\LSTM\backbone_neck.onnx")
        output_dir = Path("extracted_features_pt")

    # Đường dẫn file metadata (nếu có, mặc định tự động tìm trong data_dir)
    split_file: Optional[Path] = None
    scan_folders: bool = False         # True: Bắt buộc quét cây thư mục train/ và val/ (bỏ qua file json)
    video_exts: Tuple[str, ...] = (".mp4", ".avi", ".mkv", ".mov")

    # Tên file tensor đầu ra (.pt)
    train_name: str = "features_merged_train.pt"
    val_name: str = "features_merged_val.pt"
    active_splits: Tuple[str, ...] = ("train", "val")

    # ---------------- 2. THAM SỐ LẤY MẪU CHUỖI & ẢNH (TỐI ƯU HÓA) ----------------
    sample_interval: float = 0.1    # Khoảng thời gian t (giây) giữa 2 lần lấy mẫu (Target FPS = 10.0, ~100 frames/clip)
    seq_len: Optional[int] = None   # None: Chuỗi động tự nhiên (Cách 1 - Chuẩn); int: Cố định độ dài (Cách 2)
    img_size: int = 640             # Kích thước letterbox (640x640 cho backbone_neck.onnx)
    chunk_size: int = 100            # Kích thước mini-chunk forward ONNX (tối ưu thông lượng PCIe & Tensor Cores)
    use_fp16: bool = True           # Khuyến nghị True: Lưu float16 tiết kiệm 50% RAM & Disk

    # ---------------- 3. TỐI ƯU HÓA BỘ NHỚ RAM & ĐÓNG GÓI ----------------
    gc_interval: int = 2           # Thu gom rác gc.collect() & empty_cache() mỗi 50 clips
    shard_size: Optional[int] = 5000 # Phân mảnh 5,000 mẫu/shard cho tập Train (19,540 mẫu -> 4 shards an toàn)
    skip_package: bool = False      # True: Chỉ lưu cache, bỏ qua đóng gói tệp .pt
    force_recompute: bool = False   # True: Bỏ qua cache hiện có, trích xuất lại từ đầu

    # ---------------- 4. TĂNG CƯỜNG DỮ LIỆU (DATA AUGMENTATION) ----------------
    augment: bool = True            # Bật/tắt tăng cường dữ liệu
    num_aug: int = 4                # Số bản sao tăng cường cho mỗi video train (1 gốc + 4 augment = 5 bản)
    include_original: bool = True   # Giữ lại video gốc bên cạnh bản augment
    augment_splits: Tuple[str, ...] = ("train",)  # Chỉ tăng cường cho tập train
    aug_seed: int = 42              # Seed cơ sở ngẫu nhiên cho augment

    # ---------------- 5. CẤU HÌNH ĐA GPU KAGGLE T4x2 ----------------
    # Sử dụng toàn bộ GPU CUDA phát hiện được ([0, 1] trên T4x2, fallback [0])
    device_ids: List[int] = list(AVAILABLE_CUDA_DEVICES) if AVAILABLE_CUDA_DEVICES else [0]
    cudnn_algo: str = "EXHAUSTIVE"     # cuDNN conv search: EXHAUSTIVE, HEURISTIC, DEFAULT
    limit: Optional[int] = None     # Giới hạn số lượng clip mỗi tập để chạy thử nghiệm (None = toàn bộ)

    @property
    def cache_dir(self) -> Path:
        return self.output_dir / "cache"

cfg = KaggleExtractConfig()

print("=" * 80)
print("                   CẤU HÌNH TIẾN TRÌNH TRÍCH XUẤT ĐẶC TRƯNG                   ")
print("=" * 80)
print(f"[*] Môi trường chạy     : {'Kaggle Notebook' if IS_KAGGLE else 'Local Environment'}")
print(f"[*] Dataset Root Dir    : {cfg.data_dir} (Tồn tại: {cfg.data_dir.exists()})")
print(f"[*] ONNX Model Path     : {cfg.onnx_path} (Tồn tại: {cfg.onnx_path.exists()})")
print(f"[*] Output Directory    : {cfg.output_dir}")
print(f"[*] Cache Directory     : {cfg.cache_dir}")
print(f"[*] Active Splits       : {cfg.active_splits}")
print(f"[*] Sample Interval     : {cfg.sample_interval}s (Target FPS = {1.0 / cfg.sample_interval:.1f})")
print(f"[*] Sequence Length     : {cfg.seq_len if cfg.seq_len is not None else 'None (Cách 1: List[Tensor] chuỗi động tự nhiên)'}")
print(f"[*] Image Size          : {cfg.img_size}x{cfg.img_size}")
print(f"[*] Chunk Size          : {cfg.chunk_size}")
print(f"[*] Precision           : {'Float16 (Tiết kiệm 50% Disk/RAM)' if cfg.use_fp16 else 'Float32'}")
print(f"[*] Data Augmentation   : {'BẬT (Số bản=' + str(cfg.num_aug) + ')' if cfg.augment else 'TẮT'}")
print(f"[*] Thiết bị GPU xử lý  : {cfg.device_ids} ({len(cfg.device_ids)} GPU song song)")
print(f"[*] cuDNN Conv Algo     : {cfg.cudnn_algo}")
print(f"[*] GC Interval         : Mỗi {cfg.gc_interval} clips")
print(f"[*] Shard Size          : {cfg.shard_size if cfg.shard_size else 'Không phân mảnh'}")
print(f"[*] Test Mode (Limit)   : {cfg.limit if cfg.limit else 'Không giới hạn (Toàn bộ dataset)'}")
print("=" * 80)


In [ ]:
# ==============================================================================
# CELL 4: MODULE TĂNG CƯỜNG DỮ LIỆU ĐỒNG BỘ THỜI GIAN (AUGMENTATION MODULE)
# ==============================================================================
# Tạo tệp augment.py trực tiếp tại thư mục làm việc để các tiến trình con multiprocessing
# có thể import độc lập một cách an toàn mà không bị lỗi Pickling/ModuleNotFound.
augment_script_path = Path("augment.py")

augment_code = """import inspect
import os
import sys
import random
from typing import List, Tuple, Optional

if hasattr(sys.stdout, "reconfigure"): sys.stdout.reconfigure(encoding="utf-8")
if hasattr(sys.stderr, "reconfigure"): sys.stderr.reconfigure(encoding="utf-8")
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import albumentations as A
import cv2
import numpy as np

def _make_gauss_noise(var_limit, p):
    params = inspect.signature(A.GaussNoise.__init__).parameters
    if "var_limit" in params:
        return A.GaussNoise(var_limit=var_limit, p=p)
    if "std_range" in params:
        var_min, var_max = var_limit
        std_min = max(0.0, min(1.0, (var_min ** 0.5) / 255.0))
        std_max = max(0.0, min(1.0, (var_max ** 0.5) / 255.0))
        return A.GaussNoise(std_range=(std_min, std_max), p=p)
    raise RuntimeError("Phiên bản albumentations không hỗ trợ tham số GaussNoise.")

def _make_shift_scale_rotate(shift_limit, scale_limit, rotate_limit, p, fill_color=(114, 114, 114)):
    if hasattr(A, "Affine"):
        scale = (1.0 - scale_limit, 1.0 + scale_limit) if isinstance(scale_limit, (int, float)) else (1.0 + scale_limit[0], 1.0 + scale_limit[1])
        translate = (-shift_limit, shift_limit) if isinstance(shift_limit, (int, float)) else shift_limit
        rotate = (-rotate_limit, rotate_limit) if isinstance(rotate_limit, (int, float)) else rotate_limit
        return A.Affine(scale=scale, translate_percent=translate, rotate=rotate, border_mode=cv2.BORDER_CONSTANT, fill=fill_color, p=p)
    params = inspect.signature(A.ShiftScaleRotate.__init__).parameters
    kwargs = dict(shift_limit=shift_limit, scale_limit=scale_limit, rotate_limit=rotate_limit, border_mode=cv2.BORDER_CONSTANT, p=p)
    kwargs["value" if "value" in params else "fill"] = fill_color
    return A.ShiftScaleRotate(**kwargs)

class DetectionAugmenter:
    # Tăng cường dữ liệu video đảm bảo tính nhất quán thời gian (dùng chung seed cho mọi frame cùng clip)
    def __init__(self, cfg: dict):
        self.cfg = cfg
        shift_limit, scale_limit, rotate_limit, ssr_p = cfg["shiftScaleRotate"]
        hue_shift, sat_shift, val_shift, hsv_p = cfg["hueSaturationValue"]
        var_min, var_max, gn_p = cfg["gaussNoise"]
        blur_limit, blur_p = cfg["blur"]
        p = cfg["p"]

        self.transform = A.Compose(
            [
                A.HorizontalFlip(p=cfg["horizontalFlip"]),
                _make_shift_scale_rotate(shift_limit, scale_limit, rotate_limit, ssr_p),
                A.RandomBrightnessContrast(p=cfg["randomBrightnessContrast"]),
                A.HueSaturationValue(hue_shift_limit=hue_shift, sat_shift_limit=sat_shift, val_shift_limit=val_shift, p=hsv_p),
                _make_gauss_noise((var_min, var_max), p=gn_p),
                A.Blur(blur_limit=blur_limit, p=blur_p),
            ],
            p=p
        )
        self.video_seed: Optional[int] = None

    def set_seed(self, seed: int):
        random.seed(seed)
        np.random.seed(seed)
        if hasattr(self.transform, "set_random_seed"):
            self.transform.set_random_seed(seed)

    def __call__(self, image: np.ndarray, seed: Optional[int] = None) -> np.ndarray:
        effective_seed = seed if seed is not None else self.video_seed
        if effective_seed is not None:
            self.set_seed(effective_seed)
        try:
            out = self.transform(image=image)
            return out["image"]
        except Exception:
            return image

    def augment_video(self, frames: List[np.ndarray], boxes_list=None, labels_list=None, seed: Optional[int] = None):
        if seed is None:
            seed = random.randint(0, 2 ** 31 - 1)
        aug_frames = []
        for frame in frames:
            aug_f = self(frame, seed=seed)
            aug_frames.append(aug_f)
        return aug_frames, None, None, seed

config = {
    "horizontalFlip": 0.5,
    "shiftScaleRotate": (0.08, 0.10, 15, 0.8),
    "randomBrightnessContrast": 0.4,
    "hueSaturationValue": (15, 20, 15, 0.3),
    "gaussNoise": (5.0, 25.0, 0.25),
    "blur": (5, 0.15),
    "p": 0.5
}
"""

with open(augment_script_path, "w", encoding="utf-8") as f:
    f.write(augment_code)

import augment
from augment import DetectionAugmenter, config as DEFAULT_AUG_CONFIG
print(f"[+] Khởi tạo thành công module tăng cường dữ liệu: {augment_script_path.resolve()}")

In [ ]:
# ==============================================================================
# CELL 5: QUẢN LÝ DỮ LIỆU THEO CẤU TRÚC CHUẨN (STRUCT_DATASET.MD)
# ==============================================================================
@dataclass
class VideoClipItem:
    video_path: Path
    label: int           # 0: Alert / Driving, 1: Drowsy / Drowsiness
    clip_id: str
    video_id: str
    split: str           # 'train' hoặc 'val'
    dataset: str = "unknown"  # 'sust', 'uta-rldd', 'vbddd', etc.
    subject_id: str = "unknown"

def infer_dataset_source(clip_name: str) -> str:
    # Suy đoán tập dữ liệu gốc từ tiền tố tên clip
    name_l = clip_name.lower()
    if name_l.startswith("sust"):
        return "sust"
    if name_l.startswith("uta"):
        return "uta-rldd"
    if name_l.startswith("vbddd"):
        return "vbddd"
    return "other"

def resolve_label_from_folder_or_name(item_path: Path) -> int:
    # Xác định nhãn nhị phân dựa vào tên thư mục cha hoặc tên tệp
    parent = item_path.parent.name.lower()
    if parent.startswith("0") or "alert" in parent or "normal" in parent or "driving" in parent:
        return 0
    if parent.startswith("1") or "drowsy" in parent or "drowsiness" in parent:
        return 1

    stem = item_path.stem.lower()
    if "drowsiness" in stem or "drowsy" in stem or stem.startswith("d_") or stem == "d" or "_label10_" in stem or "_label1_" in stem:
        return 1
    if "driving" in stem or "alert" in stem or "normal" in stem or stem.startswith("n_") or stem == "n" or "_label0_" in stem:
        return 0
    raise ValueError(f"Không thể xác định nhãn từ thư mục cha hoặc tên video: {item_path}")

def load_dataset_from_json(json_path: Path, data_root: Path) -> Tuple[List[VideoClipItem], List[VideoClipItem], Dict[str, Any]]:
    # Nạp danh sách clip Train và Validation từ tệp metadata dataset_merged_split.json
    with open(json_path, "r", encoding="utf-8") as f:
        meta_data = json.load(f)

    train_clips_raw = meta_data.get("train_clips", [])
    val_clips_raw = meta_data.get("val_clips", [])

    train_items: List[VideoClipItem] = []
    val_items: List[VideoClipItem] = []

    for c in train_clips_raw:
        rel_p = c.get("processed_path", "")
        vid_path = data_root / rel_p
        if not vid_path.exists():
            alt_path = data_root / "train" / ("1_drowsy" if c.get("binary_label", 0) == 1 else "0_alert") / Path(rel_p).name
            if alt_path.exists():
                vid_path = alt_path

        train_items.append(VideoClipItem(
            video_path=vid_path,
            label=int(c.get("binary_label", 0)),
            clip_id=str(c.get("clip_id", vid_path.stem)),
            video_id=vid_path.stem,
            split="train",
            dataset=str(c.get("dataset", infer_dataset_source(vid_path.name))),
            subject_id=str(c.get("subject_id", "unknown"))
        ))

    for c in val_clips_raw:
        rel_p = c.get("processed_path", "")
        vid_path = data_root / rel_p
        if not vid_path.exists():
            alt_path = data_root / "val" / ("1_drowsy" if c.get("binary_label", 0) == 1 else "0_alert") / Path(rel_p).name
            if alt_path.exists():
                vid_path = alt_path

        val_items.append(VideoClipItem(
            video_path=vid_path,
            label=int(c.get("binary_label", 0)),
            clip_id=str(c.get("clip_id", vid_path.stem)),
            video_id=vid_path.stem,
            split="val",
            dataset=str(c.get("dataset", infer_dataset_source(vid_path.name))),
            subject_id=str(c.get("subject_id", "unknown"))
        ))

    return train_items, val_items, meta_data

def scan_dataset_from_directory(data_root: Path, video_exts: Tuple[str, ...]) -> Tuple[List[VideoClipItem], List[VideoClipItem], Dict[str, Any]]:
    # Quét trực tiếp cấu trúc thư mục train/ và val/ (0_alert vs 1_drowsy)
    norm_exts = {e.lower() if e.startswith(".") else f".{e.lower()}" for e in video_exts}

    def _scan_split_folder(split_folder: Path, split_name: str) -> List[VideoClipItem]:
        items = []
        if not split_folder.exists():
            return items
        for sub_dir in sorted(split_folder.iterdir()):
            if not sub_dir.is_dir():
                continue
            sub_name = sub_dir.name.lower()
            if sub_name.startswith("0") or "alert" in sub_name or "normal" in sub_name or "driving" in sub_name:
                sub_label = 0
            elif sub_name.startswith("1") or "drowsy" in sub_name or "drowsiness" in sub_name:
                sub_label = 1
            else:
                continue

            for vf in sorted(sub_dir.rglob("*")):
                if vf.is_file() and vf.suffix.lower() in norm_exts:
                    items.append(VideoClipItem(
                        video_path=vf,
                        label=sub_label,
                        clip_id=vf.stem,
                        video_id=vf.stem,
                        split=split_name,
                        dataset=infer_dataset_source(vf.name),
                        subject_id=vf.stem.split("-")[0] if "-" in vf.stem else vf.stem.split("_")[0]
                    ))
        return items

    train_folder = data_root / "train"
    val_folder = data_root / "val" if (data_root / "val").exists() else data_root / "validation"

    train_items = _scan_split_folder(train_folder, "train")
    val_items = _scan_split_folder(val_folder, "val")

    metadata = {
        "source": "folder_scan",
        "train_clips_count": len(train_items),
        "val_clips_count": len(val_items),
        "total_clips": len(train_items) + len(val_items),
    }
    return train_items, val_items, metadata

def prepare_structured_dataset(cfg: KaggleExtractConfig) -> Tuple[List[VideoClipItem], List[VideoClipItem], Dict[str, Any]]:
    # Điểm vào chuẩn bị dataset: ưu tiên JSON metadata, fallback quét thư mục
    if not cfg.data_dir.exists():
        raise FileNotFoundError(f"Thư mục dataset không tồn tại: {cfg.data_dir.resolve()}")

    target_json = cfg.split_file if (cfg.split_file and cfg.split_file.exists()) else (cfg.data_dir / "dataset_merged_split.json")

    if not cfg.scan_folders and target_json.exists():
        print(f"[+] Tìm thấy tệp metadata JSON: {target_json.resolve()}")
        try:
            train_items, val_items, meta = load_dataset_from_json(target_json, cfg.data_dir)
            existing_count = sum(1 for item in train_items[:10] if item.video_path.exists())
            if existing_count > 0:
                print(f"[+] Nạp thành công từ JSON: {len(train_items)} train clips, {len(val_items)} val clips.")
                return train_items, val_items, meta
            else:
                print(f"[WARN] Đường dẫn trong JSON không khớp tệp trên đĩa, chuyển sang quét cây thư mục...")
        except Exception as e:
            print(f"[WARN] Lỗi đọc JSON ({e}), chuyển sang quét cây thư mục...")

    print(f"[*] Đang quét cây thư mục dữ liệu tại: {cfg.data_dir.resolve()}...")
    train_items, val_items, meta = scan_dataset_from_directory(cfg.data_dir, cfg.video_exts)
    print(f"[+] Quét hoàn tất: {len(train_items)} train clips, {len(val_items)} val clips.")

    if len(train_items) == 0 and len(val_items) == 0:
        raise FileNotFoundError(
            f"Không tìm thấy video clip nào trong {cfg.data_dir.resolve()}.\n"
            f"Vui lòng đảm bảo cấu trúc thư mục:\n"
            f"  {cfg.data_dir / 'train' / '0_alert'}\n"
            f"  {cfg.data_dir / 'train' / '1_drowsy'}\n"
            f"  {cfg.data_dir / 'val' / '0_alert'}\n"
            f"  {cfg.data_dir / 'val' / '1_drowsy'}"
        )
    return train_items, val_items, meta

# Thực thi quét dataset
train_items, val_items, dataset_meta = prepare_structured_dataset(cfg)

if cfg.limit:
    train_items = train_items[:cfg.limit]
    val_items = val_items[:cfg.limit]
    print(f"[!] TEST MODE: Giới hạn xử lý tối đa {cfg.limit} clip mỗi tập (Train={len(train_items)}, Val={len(val_items)}).")

num_train_alert = sum(1 for x in train_items if x.label == 0)
num_train_drowsy = sum(1 for x in train_items if x.label == 1)
num_val_alert = sum(1 for x in val_items if x.label == 0)
num_val_drowsy = sum(1 for x in val_items if x.label == 1)

print("-" * 80)
print(f"[*] THỐNG KÊ TẬP DỮ LIỆU:")
print(f"    - TẬP TRAIN: {len(train_items)} clips | Alert (0) = {num_train_alert}, Drowsy (1) = {num_train_drowsy}")
print(f"    - TẬP VAL  : {len(val_items)} clips | Alert (0) = {num_val_alert}, Drowsy (1) = {num_val_drowsy}")
print("-" * 80)

In [ ]:
# ==============================================================================
# CELL 6: MODULE TIẾN TRÌNH CON ĐA GPU (MULTI-GPU WORKER SCRIPT)
# ==============================================================================
# Để chạy multiprocessing qua spawn trên Kaggle / Linux / Windows mà không gặp lỗi
# PicklingError hay CUDA context conflict, toàn bộ logic của worker được tách vào file
# multi_gpu_worker.py độc lập.
worker_file_path = Path("multi_gpu_worker.py")

worker_module_code = """import os, sys, time, math, hashlib, gc
from pathlib import Path
from typing import List, Tuple, Optional, Dict, Any

if hasattr(sys.stdout, "reconfigure"): sys.stdout.reconfigure(encoding="utf-8")
if hasattr(sys.stderr, "reconfigure"): sys.stderr.reconfigure(encoding="utf-8")
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
import torch.nn.functional as F
import numpy as np
import cv2
import onnxruntime as ort

from augment import DetectionAugmenter, config as DEFAULT_AUG_CONFIG

def letterbox(image: np.ndarray, new_size: int = 640, color=(114, 114, 114)) -> np.ndarray:
    # Resize ảnh giữ nguyên tỉ lệ (aspect ratio) với padding đồng màu (640x640)
    h, w = image.shape[:2]
    scale = min(new_size / h, new_size / w)
    new_w = max(1, int(round(w * scale)))
    new_h = max(1, int(round(h * scale)))
    resized = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_LINEAR)

    canvas = np.full((new_size, new_size, 3), color, dtype=image.dtype)
    pad_left = (new_size - new_w) // 2
    pad_top = (new_size - new_h) // 2
    canvas[pad_top: pad_top + new_h, pad_left: pad_left + new_w] = resized
    return canvas

class CloneONNXCUDARuntime:
    # Quản lý suy luận ONNX Runtime trên GPU CUDA với I/O Binding Zero-Copy
    def __init__(self, onnx_model_path: str, device_id: int = 0, cudnn_algo: str = "EXHAUSTIVE"):
        self.onnx_model_path = str(onnx_model_path)
        self.device_id = device_id
        self.cudnn_algo = str(cudnn_algo).upper()

        if not os.path.exists(self.onnx_model_path):
            raise FileNotFoundError(f"Không tìm thấy file ONNX: {self.onnx_model_path}")

        self.session_options = ort.SessionOptions()
        self.session_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
        self.session_options.log_severity_level = 3

        cuda_options = {
            "device_id": self.device_id,
            "arena_extend_strategy": "kNextPowerOfTwo",
            "cudnn_conv_algo_search": self.cudnn_algo,
            "cudnn_conv_use_max_workspace": "1",
            "do_copy_in_default_stream": True,
        }
        providers = [
            ("CUDAExecutionProvider", cuda_options),
            "CPUExecutionProvider"
        ]

        self.session = ort.InferenceSession(self.onnx_model_path, sess_options=self.session_options, providers=providers)
        active = self.session.get_providers()
        self.use_cuda = "CUDAExecutionProvider" in active

        input_meta = self.session.get_inputs()[0]
        self.input_name = input_meta.name
        self.expected_h = input_meta.shape[2] if len(input_meta.shape) > 2 and isinstance(input_meta.shape[2], int) else None
        self.expected_w = input_meta.shape[3] if len(input_meta.shape) > 3 and isinstance(input_meta.shape[3], int) else None
        self.output_names = [o.name for o in self.session.get_outputs()]

    def forward(self, input_tensor: torch.Tensor) -> List[torch.Tensor]:
        # Zero-copy CUDA I/O Binding sang PyTorch CUDA Tensors
        if self.use_cuda:
            if not input_tensor.is_cuda:
                input_tensor = input_tensor.to(f"cuda:{self.device_id}", non_blocking=True)
            if not input_tensor.is_contiguous():
                input_tensor = input_tensor.contiguous()

            io_binding = self.session.io_binding()
            io_binding.bind_input(
                name=self.input_name,
                device_type="cuda",
                device_id=self.device_id,
                element_type=np.float32,
                shape=tuple(input_tensor.shape),
                buffer_ptr=input_tensor.data_ptr(),
            )
            for out_name in self.output_names:
                io_binding.bind_output(out_name, device_type="cuda", device_id=self.device_id)

            self.session.run_with_iobinding(io_binding)
            raw_outputs = io_binding.get_outputs()
            return [torch.from_dlpack(out) for out in raw_outputs]
        else:
            np_in = input_tensor.detach().cpu().numpy()
            if not np_in.flags.c_contiguous:
                np_in = np.ascontiguousarray(np_in)
            outs = self.session.run(self.output_names, {self.input_name: np_in})
            return [torch.from_numpy(out) for out in outs]

def read_and_sample_video_frames(
    video_path: Path,
    seq_len: Optional[int] = None,
    sample_interval: float = 0.1,
    img_size: int = 640
) -> Tuple[List[np.ndarray], int]:
    # Đọc video và lấy mẫu tuần tự theo chu kỳ sample_interval (mặc định 0.1s - Target FPS 10.0). Letterbox về img_size
    # Tối ưu hóa: Stream decoding một chiều loại bỏ hoàn toàn cap.set overhead
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Không thể mở video: {video_path}")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = float(cap.get(cv2.CAP_PROP_FPS))
    if fps <= 0 or math.isnan(fps) or math.isinf(fps):
        fps = 30.0

    frame_step = max(sample_interval * fps, 1.0)
    frames = []

    if total_frames > 0:
        indices = []
        k = 0
        while True:
            idx = int(round(k * frame_step))
            if idx >= total_frames:
                break
            indices.append(idx)
            k += 1
            if seq_len is not None and len(indices) >= seq_len:
                break

        if len(indices) == 0 and total_frames > 0:
            indices = [0]

        indices_set = set(indices)
        max_target_idx = max(indices) if indices else 0

        cur_idx = 0
        while cur_idx <= max_target_idx:
            ret, frame = cap.read()
            if not ret:
                break
            if cur_idx in indices_set:
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frames.append(letterbox(rgb, new_size=img_size))
            cur_idx += 1
        cap.release()

        actual_frames = len(frames)
        if actual_frames == 0:
            actual_frames = 1
            frames = [np.zeros((img_size, img_size, 3), dtype=np.uint8)]

        if seq_len is not None:
            while len(frames) < seq_len:
                frames.append(frames[-1].copy())
            frames = frames[:seq_len]

        return frames, actual_frames

    # Fallback nếu total_frames <= 0 (stream không rõ độ dài)
    raw_frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        raw_frames.append(frame)
    cap.release()

    n_raw = len(raw_frames)
    if n_raw == 0:
        raise ValueError(f"Video rỗng: {video_path}")

    indices = []
    k = 0
    while True:
        idx = int(round(k * frame_step))
        if idx >= n_raw:
            break
        indices.append(idx)
        k += 1
        if seq_len is not None and len(indices) >= seq_len:
            break

    for idx in indices:
        rgb = cv2.cvtColor(raw_frames[idx], cv2.COLOR_BGR2RGB)
        frames.append(letterbox(rgb, new_size=img_size))

    del raw_frames
    actual_frames = len(frames)
    if actual_frames == 0:
        actual_frames = 1
        frames = [np.zeros((img_size, img_size, 3), dtype=np.uint8)]

    if seq_len is not None:
        while len(frames) < seq_len:
            frames.append(frames[-1].copy())
        frames = frames[:seq_len]

    return frames, actual_frames

def forward_video_chunks(
    frames_rgb: List[np.ndarray],
    runner: CloneONNXCUDARuntime,
    chunk_size: int = 64,
    device: str = "cuda:0",
    use_fp16: bool = True
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    # Forward qua ONNX theo từng mini-chunk 64 ảnh, AdaptiveAvgPool2d về 1x1 và Early FP16
    # Tối ưu hóa: Truyền mảng uint8 qua PCIe (giảm 75% tải bus) và chia 255.0 trực tiếp trên GPU CUDA Cores
    seq_len = len(frames_rgb)
    target_dtype = torch.float16 if use_fp16 else torch.float32

    if seq_len == 0:
        return torch.empty((0, 64), dtype=target_dtype), torch.empty((0, 128), dtype=target_dtype), torch.empty((0, 256), dtype=target_dtype)

    name_to_idx = {name: i for i, name in enumerate(runner.output_names)}
    idx_p3 = name_to_idx.get("p3", 0)
    idx_p4 = name_to_idx.get("p4", 1)
    idx_p5 = name_to_idx.get("p5", 2)

    target_device = device if (runner.use_cuda and torch.cuda.is_available()) else "cpu"
    p3_chunks, p4_chunks, p5_chunks = [], [], []

    for i in range(0, seq_len, chunk_size):
        chunk_frames = frames_rgb[i : i + chunk_size]
        chunk_np = np.stack([np.ascontiguousarray(f.transpose(2, 0, 1)) for f in chunk_frames], axis=0)
        chunk_tensor = torch.from_numpy(chunk_np).to(target_device, non_blocking=True)
        chunk_tensor = chunk_tensor.float().div_(255.0)

        outs = runner.forward(chunk_tensor)

        p3_pool = F.adaptive_avg_pool2d(outs[idx_p3], (1, 1)).flatten(1).cpu()
        p4_pool = F.adaptive_avg_pool2d(outs[idx_p4], (1, 1)).flatten(1).cpu()
        p5_pool = F.adaptive_avg_pool2d(outs[idx_p5], (1, 1)).flatten(1).cpu()

        if use_fp16:
            p3_pool = p3_pool.half()
            p4_pool = p4_pool.half()
            p5_pool = p5_pool.half()

        p3_chunks.append(p3_pool)
        p4_chunks.append(p4_pool)
        p5_chunks.append(p5_pool)
        del chunk_tensor, chunk_np, outs

    p3_tensor = torch.cat(p3_chunks, dim=0)
    p4_tensor = torch.cat(p4_chunks, dim=0)
    p5_tensor = torch.cat(p5_chunks, dim=0)
    return p3_tensor, p4_tensor, p5_tensor

def gpu_worker_loop(
    worker_id: int,
    gpu_id: int,
    task_queue,
    result_queue,
    onnx_model_path: str,
    split_cache_dir_str: str,
    seq_len: Optional[int] = None,
    sample_interval: float = 0.1,
    img_size: int = 640,
    chunk_size: int = 64,
    use_fp16: bool = True,
    augment_enabled: bool = False,
    num_aug: int = 0,
    include_original: bool = True,
    base_seed: int = 42,
    force_recompute: bool = False,
    gc_interval: int = 50,
    cudnn_algo: str = "EXHAUSTIVE"
):
    # Vòng lặp worker chạy độc lập trên 1 GPU CUDA riêng biệt
    try:
        if torch.cuda.is_available():
            torch.cuda.set_device(gpu_id)
        device_str = f"cuda:{gpu_id}" if torch.cuda.is_available() else "cpu"

        runner = CloneONNXCUDARuntime(onnx_model_path=onnx_model_path, device_id=gpu_id, cudnn_algo=cudnn_algo)
        # Warmup mô hình: warm up cả chunk_size và batch dư (ví dụ 64 và 36 cho 100 frames)
        dummy_main = torch.zeros(chunk_size, 3, img_size, img_size, device=device_str)
        runner.forward(dummy_main)
        del dummy_main
        remainder = (seq_len % chunk_size) if (seq_len is not None and seq_len % chunk_size != 0) else (100 % chunk_size)
        if remainder > 0 and remainder != chunk_size:
            dummy_rem = torch.zeros(remainder, 3, img_size, img_size, device=device_str)
            runner.forward(dummy_rem)
            del dummy_rem
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        augmenter = None
        if augment_enabled and num_aug > 0:
            augmenter = DetectionAugmenter(DEFAULT_AUG_CONFIG)

        split_cache_dir = Path(split_cache_dir_str)
        split_cache_dir.mkdir(parents=True, exist_ok=True)
        is_augmented = (augmenter is not None and num_aug > 0)
        actual_include_original = include_original if is_augmented else True

        result_queue.put({"type": "ready", "worker_id": worker_id, "gpu_id": gpu_id})
        processed_count = 0

        while True:
            task = task_queue.get()
            if task is None:
                break

            video_path_str, label, clip_id, video_id, dataset, subject_id = task
            video_path = Path(video_path_str)

            targets = []
            if actual_include_original:
                targets.append({"sub_id": video_id, "is_aug": False, "aug_idx": 0, "seed": None})
            if is_augmented:
                for k in range(1, num_aug + 1):
                    aug_id = f"{video_id}_aug{k}"
                    aug_seed = (base_seed + int(hashlib.md5(aug_id.encode("utf-8")).hexdigest()[:8], 16)) % (2 ** 31 - 1)
                    targets.append({"sub_id": aug_id, "is_aug": True, "aug_idx": k, "seed": aug_seed})

            needed_targets = []
            for t in targets:
                cache_file = split_cache_dir / f"{t['sub_id']}.pt"
                if force_recompute or not cache_file.exists():
                    needed_targets.append(t)
                else:
                    try:
                        cdata = torch.load(cache_file, map_location="cpu")
                        c_interval = cdata.get("sample_interval", None)
                        if c_interval is not None and abs(c_interval - sample_interval) > 1e-4:
                            needed_targets.append(t)
                    except Exception:
                        needed_targets.append(t)

            if len(needed_targets) == 0:
                result_queue.put({
                    "type": "done",
                    "worker_id": worker_id,
                    "gpu_id": gpu_id,
                    "video_id": video_id,
                    "status": "cached",
                    "targets": [t["sub_id"] for t in targets]
                })
                continue

            try:
                raw_frames, actual_frames = read_and_sample_video_frames(
                    video_path=video_path,
                    seq_len=seq_len,
                    sample_interval=sample_interval,
                    img_size=img_size
                )

                for t in needed_targets:
                    if t["is_aug"]:
                        aug_frames, _, _, _ = augmenter.augment_video(frames=raw_frames, seed=t["seed"])
                        p3, p4, p5 = forward_video_chunks(aug_frames, runner, chunk_size=chunk_size, device=device_str, use_fp16=use_fp16)
                        del aug_frames
                    else:
                        p3, p4, p5 = forward_video_chunks(raw_frames, runner, chunk_size=chunk_size, device=device_str, use_fp16=use_fp16)

                    cache_file = split_cache_dir / f"{t['sub_id']}.pt"
                    torch.save({
                        "video_id": t["sub_id"],
                        "clip_id": clip_id,
                        "label": label,
                        "p3": p3,
                        "p4": p4,
                        "p5": p5,
                        "actual_frames": actual_frames,
                        "sample_interval": sample_interval,
                        "is_augmented": t["is_aug"],
                        "aug_seed": t["seed"],
                        "dataset": dataset,
                        "subject_id": subject_id
                    }, cache_file)
                    del p3, p4, p5

                del raw_frames
                processed_count += 1

                if processed_count % gc_interval == 0:
                    gc.collect()
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()

                result_queue.put({
                    "type": "done",
                    "worker_id": worker_id,
                    "gpu_id": gpu_id,
                    "video_id": video_id,
                    "status": "extracted",
                    "targets": [t["sub_id"] for t in targets]
                })

            except Exception as e:
                result_queue.put({
                    "type": "error",
                    "worker_id": worker_id,
                    "gpu_id": gpu_id,
                    "video_id": video_id,
                    "error": str(e)
                })

    except Exception as fatal_e:
        result_queue.put({
            "type": "fatal_error",
            "worker_id": worker_id,
            "gpu_id": gpu_id,
            "error": str(fatal_e)
        })
"""

with open(worker_file_path, "w", encoding="utf-8") as f:
    f.write(worker_module_code)

import multi_gpu_worker
print(f"[+] Khởi tạo thành công module worker đa GPU: {worker_file_path.resolve()}")


In [ ]:
# ==============================================================================
# CELL 7: BỘ ĐIỀU PHỐI ĐA GPU (MULTI-GPU ENGINE) & ĐÓNG GÓI TENSOR (CONSOLIDATION)
# ==============================================================================
def consolidate_split_features(
    split_name: str,
    cache_files: List[Path],
    output_pt_path: Path,
    seq_len: Optional[int] = None,
    sample_interval: float = 0.1,
    use_fp16: bool = True,
    is_augmented_split: bool = False,
    num_aug: int = 0,
    actual_include_original: bool = True,
    shard_size: Optional[int] = 5000
) -> None:
    # Pha 2: Đóng gói các đặc trưng từ cache trên đĩa thành tệp .pt hoàn chỉnh
    if len(cache_files) == 0:
        print(f"[WARN] Không có tệp cache nào để đóng gói cho tập {split_name.upper()}!")
        return

    print(f"\n{'-' * 80}")
    print(f"[*] BẮT ĐẦU PHA 2: ĐÓNG GÓI TENSOR CHO TẬP: {split_name.upper()} ({len(cache_files)} MẪU)")
    print(f"[*] Đường dẫn xuất: {output_pt_path.resolve()}")
    print(f"[*] Chế độ phân mảnh: {'BẬT (Shard size = ' + str(shard_size) + ')' if shard_size else 'TẮT (Gộp 1 file đơn lẻ)'}")
    print(f"{'-' * 80}")

    def _package_single_shard(sub_files: List[Path], target_path: Path, part_idx: Optional[int] = None) -> Dict[str, Any]:
        p3_list, p4_list, p5_list = [], [], []
        labels_list, video_ids_list, seq_lens_list = [], [], []
        datasets_list, subjects_list, clip_ids_list = [], [], []

        desc_label = f"[Đóng gói {split_name.upper()}{f' Part {part_idx}' if part_idx is not None else ''}]"
        for cf in tqdm(sub_files, desc=desc_label):
            if not cf.exists():
                continue
            cdata = torch.load(cf, map_location="cpu")
            p3, p4, p5 = cdata["p3"], cdata["p4"], cdata["p5"]

            if use_fp16:
                if p3.dtype != torch.float16: p3 = p3.half()
                if p4.dtype != torch.float16: p4 = p4.half()
                if p5.dtype != torch.float16: p5 = p5.half()

            p3_list.append(p3)
            p4_list.append(p4)
            p5_list.append(p5)
            labels_list.append(cdata["label"])
            video_ids_list.append(cdata["video_id"])
            clip_ids_list.append(cdata.get("clip_id", cdata["video_id"]))
            seq_lens_list.append(cdata.get("actual_frames", p3.shape[0]))
            if "dataset" in cdata:
                datasets_list.append(cdata["dataset"])
            if "subject_id" in cdata:
                subjects_list.append(cdata["subject_id"])
            del cdata

        if len(video_ids_list) == 0:
            return {}

        final_labels = torch.tensor(labels_list, dtype=torch.long)
        final_seq_lens = torch.tensor(seq_lens_list, dtype=torch.int32)

        if seq_len is None:
            final_p3 = p3_list  # List[Tensor [T_i, 64]]
            final_p4 = p4_list  # List[Tensor [T_i, 128]]
            final_p5 = p5_list  # List[Tensor [T_i, 256]]
            is_var_len = True
        else:
            final_p3 = torch.stack(p3_list, dim=0)  # [N, seq_len, 64]
            final_p4 = torch.stack(p4_list, dim=0)  # [N, seq_len, 128]
            final_p5 = torch.stack(p5_list, dim=0)  # [N, seq_len, 256]
            is_var_len = False

        save_dict = {
            "p3": final_p3,
            "p4": final_p4,
            "p5": final_p5,
            "labels": final_labels,
            "video_ids": video_ids_list,
            "clip_ids": clip_ids_list,
            "seq_lens": final_seq_lens,
            "is_variable_len": is_var_len,
            "sample_interval": sample_interval,
            "dtype": "float16" if use_fp16 else "float32",
            "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
            "augmented": is_augmented_split,
            "num_aug": num_aug if is_augmented_split else 0,
            "include_original": actual_include_original,
            "datasets": datasets_list,
            "subjects": subjects_list
        }

        target_path.parent.mkdir(parents=True, exist_ok=True)
        torch.save(save_dict, str(target_path))

        num_alerts = (final_labels == 0).sum().item()
        num_drowsy = (final_labels == 1).sum().item()
        num_orig = sum(1 for vid in video_ids_list if "_aug" not in vid)
        num_aug_count = sum(1 for vid in video_ids_list if "_aug" in vid)
        file_size_mb = target_path.stat().st_size / (1024 * 1024)

        print(f"[SUCCESS] Đã lưu tệp: {target_path.resolve()} ({file_size_mb:.2f} MB)")
        print(f"          - Tổng số mẫu: {len(video_ids_list)} (Gốc: {num_orig}, Augment: {num_aug_count})")
        print(f"          - Phân bố nhãn: Tỉnh táo (0) = {num_alerts}, Buồn ngủ (1) = {num_drowsy}")
        if is_var_len:
            print(f"          - Cấu trúc: List[{len(final_p3)}] (min={final_seq_lens.min().item()}, max={final_seq_lens.max().item()}, mean={final_seq_lens.float().mean().item():.1f} frames)")
        else:
            print(f"          - Shape: p3={list(final_p3.shape)}, p4={list(final_p4.shape)}, p5={list(final_p5.shape)}")

        del p3_list, p4_list, p5_list, final_p3, final_p4, final_p5, save_dict
        gc.collect()
        return {
            "file": str(target_path.resolve()),
            "total_samples": len(video_ids_list),
            "size_mb": file_size_mb
        }

    if shard_size and shard_size > 0 and len(cache_files) > shard_size:
        total_shards = math.ceil(len(cache_files) / shard_size)
        shards_info = []
        for s_idx in range(total_shards):
            start_i = s_idx * shard_size
            end_i = min(start_i + shard_size, len(cache_files))
            shard_path = output_pt_path.parent / f"{output_pt_path.stem}_part{s_idx + 1:03d}.pt"
            info = _package_single_shard(cache_files[start_i:end_i], shard_path, part_idx=s_idx + 1)
            shards_info.append(info)

        master_json_path = output_pt_path.parent / f"{output_pt_path.stem}_shards_index.json"
        with open(master_json_path, "w", encoding="utf-8") as f:
            json.dump({
                "split": split_name,
                "total_shards": total_shards,
                "total_samples": len(cache_files),
                "shard_size": shard_size,
                "shards": shards_info,
                "created_at": time.strftime("%Y-%m-%d %H:%M:%S")
            }, f, indent=4, ensure_ascii=False)
        print(f"[+] Đã lưu chỉ mục phân mảnh: {master_json_path.resolve()}")
    else:
        _package_single_shard(cache_files, output_pt_path)


def run_multi_gpu_extraction(
    split_name: str,
    items: List[VideoClipItem],
    cfg: KaggleExtractConfig,
    augment_this_split: bool = False
) -> List[Path]:
    # Điều phối trích xuất đặc trưng song song trên tất cả các GPU CUDA (T4x2).
    # Sử dụng hàng đợi công việc Task Queue và giám sát thời gian thực qua tqdm.
    device_ids = cfg.device_ids
    num_workers = len(device_ids)
    split_cache_dir = cfg.cache_dir / split_name
    split_cache_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "=" * 80)
    print(f"[*] BẮT ĐẦU PHA 1: TRÍCH XUẤT ĐA GPU CHO TẬP: {split_name.upper()} ({len(items)} CLIPS)")
    print(f"[*] Số GPU tham gia: {num_workers} ({['CUDA:' + str(i) for i in device_ids]})")
    print(f"[*] Tăng cường dữ liệu: {'BẬT (num_aug=' + str(cfg.num_aug) + ')' if augment_this_split else 'TẮT'}")
    print(f"[*] cuDNN Conv Algo: {getattr(cfg, 'cudnn_algo', 'EXHAUSTIVE')}")
    print(f"[*] Thư mục Cache: {split_cache_dir.resolve()}")
    print("=" * 80)

    ctx = mp.get_context("spawn")
    task_queue = ctx.Queue()
    result_queue = ctx.Queue()

    # Nạp toàn bộ task vào queue
    for it in items:
        task_queue.put((
            str(it.video_path),
            it.label,
            it.clip_id,
            it.video_id,
            it.dataset,
            it.subject_id
        ))
    for _ in range(num_workers):
        task_queue.put(None)  # Tín hiệu kết thúc

    workers = []
    for worker_id, gpu_id in enumerate(device_ids):
        p = ctx.Process(
            target=multi_gpu_worker.gpu_worker_loop,
            kwargs=dict(
                worker_id=worker_id,
                gpu_id=gpu_id,
                task_queue=task_queue,
                result_queue=result_queue,
                onnx_model_path=str(cfg.onnx_path),
                split_cache_dir_str=str(split_cache_dir),
                seq_len=cfg.seq_len,
                sample_interval=cfg.sample_interval,
                img_size=cfg.img_size,
                chunk_size=cfg.chunk_size,
                use_fp16=cfg.use_fp16,
                augment_enabled=augment_this_split,
                num_aug=cfg.num_aug if augment_this_split else 0,
                include_original=cfg.include_original,
                base_seed=cfg.aug_seed,
                force_recompute=cfg.force_recompute,
                gc_interval=cfg.gc_interval,
                cudnn_algo=getattr(cfg, "cudnn_algo", "EXHAUSTIVE")
            )
        )
        p.start()
        workers.append(p)

    # Chờ tất cả worker gửi tín hiệu ready
    ready_count = 0
    while ready_count < num_workers:
        msg = result_queue.get()
        if msg.get("type") == "ready":
            ready_count += 1
            print(f"    -> Worker {msg['worker_id']} sẵn sàng trên GPU CUDA:{msg['gpu_id']}")
        elif msg.get("type") == "fatal_error":
            print(f"[FATAL ERROR] Worker {msg['worker_id']} trên GPU CUDA:{msg['gpu_id']} lỗi: {msg['error']}")
            for p in workers: p.terminate()
            raise RuntimeError(f"Worker khởi động thất bại: {msg['error']}")

    print(f"[+] Tất cả {num_workers} GPU Workers đã sẵn sàng! Bắt đầu trích xuất song song...")

    pbar = tqdm(total=len(items), desc=f"[{split_name.upper()} - {num_workers} GPUs]")
    cached_count, extracted_count, error_count, completed_videos = 0, 0, 0, 0
    collected_cache_files: List[Path] = []

    while completed_videos < len(items):
        msg = result_queue.get()
        msg_type = msg.get("type")

        if msg_type == "done":
            completed_videos += 1
            status = msg.get("status")
            if status == "cached":
                cached_count += 1
            else:
                extracted_count += 1

            for target_id in msg.get("targets", []):
                cf = split_cache_dir / f"{target_id}.pt"
                if cf.exists():
                    collected_cache_files.append(cf)

            ram_gb = get_current_ram_gb()
            pbar.set_postfix({
                "Mới": extracted_count,
                "Cache": cached_count,
                "Lỗi": error_count,
                "RAM": f"{ram_gb:.2f}GB"
            })
            pbar.update(1)

        elif msg_type == "error":
            completed_videos += 1
            error_count += 1
            print(f"\n[WARN] Lỗi khi xử lý video {msg.get('video_id')} (GPU CUDA:{msg.get('gpu_id')}): {msg.get('error')}")
            pbar.update(1)

        elif msg_type == "fatal_error":
            print(f"\n[FATAL] Worker {msg.get('worker_id')} (GPU CUDA:{msg.get('gpu_id')}) dừng: {msg.get('error')}")
            break

    pbar.close()

    for p in workers:
        p.join()

    print(f"[+] Hoàn tất Pha 1 ({split_name.upper()}): {completed_videos} videos | Mới: {extracted_count}, Cache: {cached_count}, Lỗi: {error_count}")
    return collected_cache_files


In [ ]:
# ==============================================================================
# CELL 8: THỰC THI TRÍCH XUẤT ĐẶC TRƯNG TOÀN DIỆN (TRAIN & VALIDATION)
# ==============================================================================
cfg.output_dir.mkdir(parents=True, exist_ok=True)
total_pipeline_start = time.time()

# 1. Lưu metadata phân bổ để phục vụ kiểm định provenance
split_info_save_path = cfg.output_dir / "dataset_split_info.json"
with open(split_info_save_path, "w", encoding="utf-8") as f:
    json.dump({
        "dataset_dir": str(cfg.data_dir.resolve()),
        "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "train_clips": len(train_items),
        "val_clips": len(val_items),
        "sample_interval": cfg.sample_interval,
        "train_samples_preview": [item.clip_id for item in train_items[:10]],
        "val_samples_preview": [item.clip_id for item in val_items[:10]],
    }, f, indent=4, ensure_ascii=False)
print(f"[+] Đã ghi nhận thông tin phân bổ tại: {split_info_save_path.resolve()}")

# 2. Xử lý tập Train
train_cache_files = []
if "train" in cfg.active_splits and len(train_items) > 0:
    should_augment_train = cfg.augment and ("train" in cfg.augment_splits)
    train_cache_files = run_multi_gpu_extraction(
        split_name="train",
        items=train_items,
        cfg=cfg,
        augment_this_split=should_augment_train
    )

    if not cfg.skip_package:
        consolidate_split_features(
            split_name="train",
            cache_files=train_cache_files,
            output_pt_path=cfg.output_dir / cfg.train_name,
            seq_len=cfg.seq_len,
            sample_interval=cfg.sample_interval,
            use_fp16=cfg.use_fp16,
            is_augmented_split=should_augment_train,
            num_aug=cfg.num_aug,
            actual_include_original=cfg.include_original,
            shard_size=cfg.shard_size
        )

# 3. Xử lý tập Validation
val_cache_files = []
if "val" in cfg.active_splits and len(val_items) > 0:
    should_augment_val = cfg.augment and ("val" in cfg.augment_splits)
    val_cache_files = run_multi_gpu_extraction(
        split_name="val",
        items=val_items,
        cfg=cfg,
        augment_this_split=should_augment_val
    )

    if not cfg.skip_package:
        consolidate_split_features(
            split_name="val",
            cache_files=val_cache_files,
            output_pt_path=cfg.output_dir / cfg.val_name,
            seq_len=cfg.seq_len,
            sample_interval=cfg.sample_interval,
            use_fp16=cfg.use_fp16,
            is_augmented_split=should_augment_val,
            num_aug=cfg.num_aug if should_augment_val else 0,
            actual_include_original=cfg.include_original,
            shard_size=cfg.shard_size
        )

total_pipeline_time = time.time() - total_pipeline_start
print("\n" + "=" * 80)
print(f"      [HOÀN TẤT GIAI ĐOẠN 1] TRÍCH XUẤT ĐẶC TRƯNG THÀNH CÔNG!     ")
print(f"      Tổng thời gian thực thi: {total_pipeline_time / 60:.2f} phút")
print("=" * 80)

In [ ]:
# ==============================================================================
# CELL 9: KIỂM ĐỊNH TÍNH TOÀN VẸN & CẤU TRÚC TENSOR (.PT) ĐẦU RA
# ==============================================================================
def inspect_pt_file(pt_path: Path, split_name: str):
    # Kiểm tra chi tiết nội dung, kích thước và phân bố nhãn của tệp tensor .pt (hỗ trợ cả file đơn lẫn shards)
    if not pt_path.exists():
        part1 = pt_path.parent / f"{pt_path.stem}_part001.pt"
        index_json = pt_path.parent / f"{pt_path.stem}_shards_index.json"
        if part1.exists():
            print("=" * 80)
            print(f"[*] PHÁT HIỆN TẬP DỮ LIỆU ĐƯỢC LƯU DẠNG PHÂN MẢNH (SHARDS): {split_name.upper()}")
            if index_json.exists():
                with open(index_json, "r", encoding="utf-8") as f:
                    idx_data = json.load(f)
                print(f"    - Chỉ mục phân mảnh   : {index_json.name}")
                print(f"    - Tổng số shards      : {idx_data.get('total_shards')}")
                print(f"    - Tổng số mẫu toàn tập: {idx_data.get('total_samples')}")
                print(f"    - Kích thước mỗi shard: {idx_data.get('shard_size')} mẫu")
            print(f"    - Đang tiến hành kiểm định Shard mẫu: {part1.name}...")
            pt_path = part1
        else:
            print(f"[!] Không tìm thấy tệp: {pt_path}")
            return

    print("=" * 80)
    print(f"[*] KIỂM ĐỊNH TẬP DỮ LIỆU TENSOR: {split_name.upper()} ({pt_path.name})")
    print(f"    - Đường dẫn : {pt_path.resolve()}")
    print(f"    - Dung lượng: {pt_path.stat().st_size / (1024 ** 2):.2f} MB")
    print("-" * 80)

    data = torch.load(pt_path, map_location="cpu")
    print(f"    - Các trường metadata  : {list(data.keys())}")
    print(f"    - Kiểu dữ liệu lưu trữ : {data.get('dtype', 'float32')}")
    print(f"    - Chuỗi động (Var-len) : {data.get('is_variable_len', False)}")
    print(f"    - Khoảng lấy mẫu       : {data.get('sample_interval', 0.1)}s")

    labels = data["labels"]
    num_total = len(labels)
    num_alert = (labels == 0).sum().item()
    num_drowsy = (labels == 1).sum().item()
    print(f"    - Tổng số mẫu          : {num_total}")
    print(f"    - Phân bố nhãn         : Alert (0) = {num_alert} ({num_alert/num_total*100:.1f}%), Drowsy (1) = {num_drowsy} ({num_drowsy/num_total*100:.1f}%)")

    seq_lens = data.get("seq_lens", None)
    if seq_lens is not None:
        print(f"    - Độ dài chuỗi (Frames): Min = {seq_lens.min().item()}, Max = {seq_lens.max().item()}, Mean = {seq_lens.float().mean().item():.1f}")

    p3, p4, p5 = data["p3"], data["p4"], data["p5"]
    if isinstance(p3, list):
        print(f"    - Cấu trúc Tensor p3   : List[Tensor], mẫu đầu tiên shape = {list(p3[0].shape)}, dtype = {p3[0].dtype}")
        print(f"    - Cấu trúc Tensor p4   : List[Tensor], mẫu đầu tiên shape = {list(p4[0].shape)}, dtype = {p4[0].dtype}")
        print(f"    - Cấu trúc Tensor p5   : List[Tensor], mẫu đầu tiên shape = {list(p5[0].shape)}, dtype = {p5[0].dtype}")
    else:
        print(f"    - Tensor p3 shape      : {list(p3.shape)}, dtype = {p3.dtype}")
        print(f"    - Tensor p4 shape      : {list(p4.shape)}, dtype = {p4.dtype}")
        print(f"    - Tensor p5 shape      : {list(p5.shape)}, dtype = {p5.dtype}")

    # Kiểm tra tính tương thích với model.py / datn4ni2.ipynb
    dim_p3 = p3[0].shape[-1] if isinstance(p3, list) else p3.shape[-1]
    dim_p4 = p4[0].shape[-1] if isinstance(p4, list) else p4.shape[-1]
    dim_p5 = p5[0].shape[-1] if isinstance(p5, list) else p5.shape[-1]
    print(f"    - Kênh không gian      : p3={dim_p3}, p4={dim_p4}, p5={dim_p5} -> Tổng kênh = {dim_p3 + dim_p4 + dim_p5}")
    if (dim_p3, dim_p4, dim_p5) == (64, 128, 256):
        print("    - Tương thích mô hình  : HOÀN TOÀN TƯƠNG THÍCH VỚI SpatialFeatureAdapter & DeepLSTMClassifier [OK]")
    else:
        print(f"    - Cảnh báo: Kênh không gian ({dim_p3}, {dim_p4}, {dim_p5}) khác chuẩn (64, 128, 256).")
    print("=" * 80)

if "train" in cfg.active_splits:
    inspect_pt_file(cfg.output_dir / cfg.train_name, "TRAIN")

if "val" in cfg.active_splits:
    inspect_pt_file(cfg.output_dir / cfg.val_name, "VALIDATION")

In [ ]:
# ==============================================================================
# CELL 10: ĐÓNG GÓI ZIP ĐỂ TẢI VỀ HOẶC LƯU OUTPUT TRÊN KAGGLE
# ==============================================================================
import shutil

zip_output_path = Path("/kaggle/working/extracted_features_pt.zip") if IS_KAGGLE else Path("extracted_features_pt.zip")

print(f"[*] Đang nén thư mục '{cfg.output_dir}' thành '{zip_output_path}'...")
# Chỉ nén các tệp .pt và .json, bỏ qua cache tạm thời để tiết kiệm dung lượng
temp_archive_dir = Path("/kaggle/working/to_zip") if IS_KAGGLE else Path("./to_zip")
temp_archive_dir.mkdir(parents=True, exist_ok=True)

for pt_file in cfg.output_dir.glob("*.pt"):
    shutil.copy2(pt_file, temp_archive_dir / pt_file.name)
for json_file in cfg.output_dir.glob("*.json"):
    shutil.copy2(json_file, temp_archive_dir / json_file.name)

archive_base = str(zip_output_path.with_suffix(""))
shutil.make_archive(archive_base, "zip", temp_archive_dir)
shutil.rmtree(temp_archive_dir, ignore_errors=True)

if zip_output_path.exists():
    zip_size_mb = zip_output_path.stat().st_size / (1024 ** 2)
    print(f"[SUCCESS] Đã tạo tệp zip thành công: {zip_output_path.resolve()} ({zip_size_mb:.2f} MB)")
    print("          Bây giờ bạn có thể tải tệp này trực tiếp từ tab 'Output' của Kaggle")
    print("          hoặc dùng làm Dataset đầu vào cho notebook huấn luyện Deep LSTM (datn4ni2.ipynb).")